In [2]:
import subprocess
subprocess.run(["pip", "install", "mne", "-q"], check=True)
subprocess.run(["pip", "install", "scipy", "-q"], check=True)

import os
import numpy as np
import warnings
warnings.filterwarnings("ignore")

import mne
from scipy.ndimage import zoom

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CLASS_NAMES = ["REST", "LEFT", "RIGHT"]

Device: cuda


In [3]:
ROOT_DIR   = "/content/drive/MyDrive/files"
SUBJECTS   = [f"S{str(i).zfill(3)}" for i in range(1, 110)]
VALID_RUNS = ["R03", "R04", "R07", "R08", "R11", "R12"]

MOTOR_CHS_TARGETS = [
    'fc1', 'fc3', 'fcz', 'fc2', 'fc4',
    'c1',  'c3',  'cz',  'c2',  'c4',
    'cp1', 'cp3', 'cpz', 'cp2', 'cp4'
]

# Window: -0.2 to 2.0 sec captures full motor imagery ERD/ERS
# N_TIMES = int((2.0 - (-0.2)) * 160) + 1 = 353
TMIN    = 0
TMAX    =  4
SFREQ   =  160
N_TIMES =  641

BATCH_SIZE = 64
EPOCHS     = 150
LR         = 0.001

In [4]:
def match_motor_channels(ch_names):
    matched = []
    for ch in ch_names:
        clean = ch.lower().replace('.', '').replace(' ', '').strip()
        if clean in MOTOR_CHS_TARGETS:
            matched.append(ch)
    return matched


In [5]:
def resample_to(arr, target_t):
    """arr: (n_epochs, n_ch, n_times) -> resample last dim to target_t"""
    if arr.shape[2] == target_t:
        return arr
    factor = target_t / arr.shape[2]
    return zoom(arr, (1, 1, factor), order=1).astype(np.float32)


In [6]:
!pip install mne

In [9]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    if "S001" in dirs:
        print(f"✅ Found: {root}")
        break
    # Also print top-level structure to see layout
    if root.count(os.sep) <= 4:
        print(root)

/kaggle/input/datasets
/kaggle/input/datasets/gamalasran
✅ Found: /kaggle/input/datasets/gamalasran/physionet-eeg-motor-movement-imagery/files


In [14]:
import os
import mne
import numpy as np

# ═════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# Kaggle dataset path (from your screenshot: /kaggle/input/.../files/)
# ═════════════════════════════════════════════════════════════════════════
ROOT_DIR   = "✅ Found: /kaggle/input/datasets/gamalasran/physionet-eeg-motor-movement-imagery/files"

SUBJECTS   = [f"S{str(i).zfill(3)}" for i in range(1, 110)]
VALID_RUNS = ["R03", "R04", "R07", "R08", "R11", "R12"]

MOTOR_CHS_TARGETS = [
    'fc1', 'fc3', 'fcz', 'fc2', 'fc4',
    'c1',  'c3',  'cz',  'c2',  'c4',
    'cp1', 'cp3', 'cpz', 'cp2', 'cp4'
]

TMIN       = 0.0
TMAX       = 4.0
SFREQ      = 160
N_TIMES    = 641    # int((4.0 - 0.0) * 160) + 1

BATCH_SIZE = 64
EPOCHS     = 150
LR         = 0.001  # ✅ Fixed: print() moved to its own line

# ═════════════════════════════════════════════════════════════════════════
# VERIFY KAGGLE PATH EXISTS

ROOT_DIR ="/kaggle/input/datasets/gamalasran/physionet-eeg-motor-movement-imagery/files"

# ═════════════════════════════════════════════════════════════════════════
# CHANNEL MATCHING
# PhysioNet EDF files store channel names as 'FC1.' 'C3.' etc. (with dot)
# ═════════════════════════════════════════════════════════════════════════
def match_motor_channels(ch_names):
    """
    Match raw channel names to MOTOR_CHS_TARGETS.
    Handles PhysioNet naming quirks: 'C3.' → 'c3', 'FC1.' → 'fc1'
    """
    matched = []
    for ch in ch_names:
        ch_clean = ch.lower().strip().rstrip('.')
        if ch_clean in MOTOR_CHS_TARGETS:
            matched.append(ch)
    return matched

# ═════════════════════════════════════════════════════════════════════════
# DATA LOADING
# ═════════════════════════════════════════════════════════════════════════
print("Loading EDF files...")

# T0=rest, T1=left fist, T2=right fist  (annotation codes in PhysioNet)
event_dict = {"T0": 1, "T1": 2, "T2": 3}  # ✅ Fixed: MNE maps annotations
                                            #    to 1-based integers, NOT 0-based

all_X   = []
all_y   = []
loaded  = 0
skipped = 0

for subj in SUBJECTS:
    for run in VALID_RUNS:

        path = f"{ROOT_DIR}/{subj}/{subj}{run}.edf"

        if not os.path.exists(path):
            continue

        try:
            raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
            raw.filter(1., 40., fir_design='firwin', verbose=False)

            # Resample if needed (some files are 160 Hz, some 128 Hz)
            if raw.info['sfreq'] != SFREQ:
                raw.resample(SFREQ, verbose=False)

            events, found_event_id = mne.events_from_annotations(
                raw, verbose=False
            )

            if len(events) < 3:
                skipped += 1
                continue

            used = match_motor_channels(raw.ch_names)

            if len(used) < 2:  # ✅ Fixed: threshold = len(MOTOR_CHS_TARGETS) min viable
                skipped += 1
                continue

            picks = mne.pick_channels(raw.info['ch_names'], include=used)

            # ✅ Fixed: use found_event_id (actual codes in this file)
            #    instead of hardcoded event_dict — avoids KeyError crashes
            epochs = mne.Epochs(
                raw, events,
                event_id  = found_event_id,
                tmin      = TMIN,
                tmax      = TMAX,
                picks     = picks,
                baseline  = None,          # ✅ Fixed: baseline=(None,0) with tmin=0
                preload   = True,          #    does nothing and raises a warning;
                verbose   = False,         #    set to None when tmin=0
            )

            X = epochs.get_data().astype(np.float32)   # (n_epochs, n_ch, n_times)
            y = epochs.events[:, 2].astype(np.int64)   # event codes

            if len(X) == 0:
                skipped += 1
                continue

            all_X.append(X)
            all_y.append(y)
            loaded += 1

        except Exception as e:
            print(f"  SKIP {subj}/{run}: {e}")
            skipped += 1

print(f"\nLoaded : {loaded} files")
print(f"Skipped: {skipped} files")

# ═════════════════════════════════════════════════════════════════════════
# CONCATENATE + SHAPE CHECK
# ═════════════════════════════════════════════════════════════════════════
if not all_X:
    raise RuntimeError("No data loaded — check ROOT_DIR and file structure.")

X = np.concatenate(all_X, axis=0)   # (total_epochs, n_channels, n_times)
y = np.concatenate(all_y, axis=0)

# Trim / pad time axis to exactly N_TIMES
if X.shape[2] > N_TIMES:
    X = X[:, :, :N_TIMES]
elif X.shape[2] < N_TIMES:
    pad = np.zeros((X.shape[0], X.shape[1], N_TIMES - X.shape[2]), dtype=np.float32)
    X = np.concatenate([X, pad], axis=2)

print(f"\n✅ X shape : {X.shape}")   # expected → (N, 15, 641)
print(f"✅ y shape : {y.shape}")
print(f"✅ Classes : {np.unique(y)}")

Loading EDF files...

Loaded : 654 files
Skipped: 0 files

✅ X shape : (19640, 15, 641)
✅ y shape : (19640,)
✅ Classes : [1 2 3]


In [20]:

if len(all_X) == 0:
    raise RuntimeError(
        "No EDF files were loaded.\n"
        f"  ROOT_DIR : {ROOT_DIR}\n"
        "  Check channel names match MOTOR_CHS_TARGETS after stripping dots."
    )

min_ch = min(x.shape[1] for x in all_X)
all_X  = [x[:, :min_ch, :] for x in all_X]
all_X  = [resample_to(x, N_TIMES) for x in all_X]

X_all = np.concatenate(all_X)
y_all = np.concatenate(all_y)
y_all = y_all - 1
print("\nDataset shape before balancing:", X_all.shape)
print("  REST :", np.sum(y_all == 0))
print("  LEFT :", np.sum(y_all == 1))
print("  RIGHT:", np.sum(y_all == 2))


Dataset shape before balancing: (19640, 15, 641)
  REST : 9845
  LEFT : 4922
  RIGHT: 4873


In [21]:
print("\n" + "=" * 60)
print("STEP 3: Combining and balancing")
print("=" * 60)
n_ch = X_all.shape[1]
n_t  = X_all.shape[2]

print(f"Raw dataset  : {X_all.shape}")
print(f"  REST : {(y_all==0).sum()}")
print(f"  LEFT : {(y_all==1).sum()}")
print(f"  RIGHT: {(y_all==2).sum()}")

# Balance: downsample REST to avg(LEFT, RIGHT) count
n_lr   = ((y_all==1).sum() + (y_all==2).sum()) // 2
n_rest = min(int(n_lr * 1.1), (y_all==0).sum())  # slight surplus ok

rng      = np.random.RandomState(42)
rest_idx = rng.choice(np.where(y_all==0)[0], n_rest, replace=False)
lr_idx   = np.where(y_all > 0)[0]
keep     = np.concatenate([rest_idx, lr_idx])
rng.shuffle(keep)

X_bal = X_all[keep]
y_bal = y_all[keep]

print(f"\nBalanced dataset: {X_bal.shape}")
print(f"  REST : {(y_bal==0).sum()}")
print(f"  LEFT : {(y_bal==1).sum()}")
print(f"  RIGHT: {(y_bal==2).sum()}")



STEP 3: Combining and balancing
Raw dataset  : (19640, 15, 641)
  REST : 9845
  LEFT : 4922
  RIGHT: 4873

Balanced dataset: (15181, 15, 641)
  REST : 5386
  LEFT : 4922
  RIGHT: 4873


In [23]:
X_norm = (X_all - X_all.mean(axis=(1, 2), keepdims=True)) / \
         (X_all.std(axis=(1, 2),  keepdims=True) + 1e-8)
X_norm = X_norm.astype(np.float32)


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_all, test_size=0.3, stratify=y_all, random_state=42
)

print(f"\nTrain: {len(X_train)}  |  Test: {len(X_test)}")


Train: 13748  |  Test: 5892


In [25]:
class EEGDataset(Dataset):
    """Shape: (1, ch, time)"""
    def __init__(self, X, y):
        self.X = torch.tensor(X[:, None, :, :], dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


train_loader = DataLoader(EEGDataset(X_train, y_train), BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(EEGDataset(X_test,  y_test),  BATCH_SIZE)

In [26]:
class EEGNet(nn.Module):
    def __init__(self, ch, t, F1=16, D=2, dropout=0.5):
        super(EEGNet, self).__init__()

        F2 = F1 * D   # 32

        # Block 1: Temporal conv + Depthwise spatial conv
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1, (1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(F1),
            nn.Conv2d(F1, F2, (ch, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 4)),
            nn.Dropout(dropout)
        )

        # Block 2: Separable conv (depthwise + pointwise)
        self.block2 = nn.Sequential(
            nn.Conv2d(F2, F2, (1, 16), padding=(0, 8), groups=F2, bias=False),
            nn.Conv2d(F2, F2, (1, 1),  bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, 8)),
            nn.Dropout(dropout)
        )

        # Block 3: Extra separable conv (no pooling)
        self.block3 = nn.Sequential(
            nn.Conv2d(F2, F2, (1, 8), padding=(0, 4), groups=F2, bias=False),
            nn.Conv2d(F2, F2, (1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.Dropout(dropout)
        )

        # Dynamically compute FC input size via dummy forward pass
        # This avoids any shape mismatch regardless of ch or t
        with torch.no_grad():
            dummy  = torch.zeros(1, 1, ch, t)
            dummy  = self.block1(dummy)
            dummy  = self.block2(dummy)
            dummy  = self.block3(dummy)
            fc_in  = dummy.flatten(start_dim=1).shape[1]

        print(f"  EEGNet FC input size: {fc_in}")
        self.fc = nn.Linear(fc_in, 3)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.fc(x.flatten(start_dim=1))



In [27]:
n_ch = X_all.shape[1]
n_t  = X_all.shape[2]

print(f"\nModel input  ->  channels: {n_ch}  |  time steps: {n_t}")

eegnet_model = EEGNet(n_ch, n_t).to(device)
with torch.no_grad():
    dummy_out = eegnet_model(torch.zeros(2, 1, n_ch, n_t).to(device))
print(f"EEGNet output shape: {dummy_out.shape}  (expected: torch.Size([2, 3]))")



Model input  ->  channels: 15  |  time steps: 641
  EEGNet FC input size: 672
EEGNet output shape: torch.Size([2, 3])  (expected: torch.Size([2, 3]))


In [28]:

loss_fn   = nn.CrossEntropyLoss(label_smoothing=0.1)
opt       = torch.optim.Adam(eegnet_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR/100)

print(f"\nTraining EEGNet  (epochs={EPOCHS}, lr={LR}, "
      f"CosineAnnealing, label_smoothing=0.1, weight_decay=1e-4)")

best_acc   = 0.0
best_state = None

for epoch in range(EPOCHS):
    eegnet_model.train()
    total_loss = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(eegnet_model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(eegnet_model.parameters(), max_norm=1.0)
        opt.step()
        total_loss += loss.item()

    scheduler.step()

    # ---- Validation each epoch to track best ----
    eegnet_model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in test_loader:
            p = eegnet_model(X.to(device)).argmax(1).cpu().numpy()
            preds.append(p)
            trues.append(y.numpy())

    preds    = np.concatenate(preds)
    trues    = np.concatenate(trues)
    val_acc  = accuracy_score(trues, preds)

    # Save best model weights
    if val_acc > best_acc:
        best_acc   = val_acc
        best_state = {k: v.clone() for k, v in eegnet_model.state_dict().items()}

    if (epoch + 1) % 20 == 0:
        current_lr = scheduler.get_last_lr()[0]
        print(f"  Epoch {epoch+1:3d}/{EPOCHS}  "
              f"loss: {total_loss/len(train_loader):.4f}  "
              f"val_acc: {val_acc:.4f}  "
              f"best: {best_acc:.4f}  "
              f"lr: {current_lr:.6f}")



Training EEGNet  (epochs=150, lr=0.001, CosineAnnealing, label_smoothing=0.1, weight_decay=1e-4)
  Epoch  20/150  loss: 0.7978  val_acc: 0.7200  best: 0.7254  lr: 0.000957
  Epoch  40/150  loss: 0.7733  val_acc: 0.7369  best: 0.7434  lr: 0.000836
  Epoch  60/150  loss: 0.7619  val_acc: 0.7403  best: 0.7495  lr: 0.000658
  Epoch  80/150  loss: 0.7587  val_acc: 0.7452  best: 0.7495  lr: 0.000453
  Epoch 100/150  loss: 0.7486  val_acc: 0.7515  best: 0.7524  lr: 0.000257
  Epoch 120/150  loss: 0.7491  val_acc: 0.7519  best: 0.7532  lr: 0.000105
  Epoch 140/150  loss: 0.7459  val_acc: 0.7517  best: 0.7541  lr: 0.000021


In [29]:
from sklearn.metrics import confusion_matrix

print("Confusion Matrix:")
print(confusion_matrix(trues, preds))
eegnet_model.load_state_dict(best_state)
eegnet_model.eval()

preds, trues = [], []
with torch.no_grad():
    for X, y in test_loader:
        p = eegnet_model(X.to(device)).argmax(1).cpu().numpy()
        preds.append(p)
        trues.append(y.numpy())

preds = np.concatenate(preds)
trues = np.concatenate(trues)
acc   = accuracy_score(trues, preds)

print(f"\n{'='*50}")
print(f"EEGNet Best Test Accuracy : {best_acc:.4f}")
print(f"EEGNet Final Test Accuracy: {acc:.4f}")
print('='*50)
print(classification_report(trues, preds, target_names=CLASS_NAMES))

print("Confusion Matrix:")
print(confusion_matrix(trues, preds))

Confusion Matrix:
[[2600  168  185]
 [ 396  897  184]
 [ 377  144  941]]

EEGNet Best Test Accuracy : 0.7541
EEGNet Final Test Accuracy: 0.7541
              precision    recall  f1-score   support

        REST       0.77      0.88      0.82      2953
        LEFT       0.75      0.60      0.67      1477
       RIGHT       0.71      0.66      0.68      1462

    accuracy                           0.75      5892
   macro avg       0.74      0.71      0.72      5892
weighted avg       0.75      0.75      0.75      5892

Confusion Matrix:
[[2595  163  195]
 [ 399  885  193]
 [ 367  132  963]]
